# Sprint 2 · Declare, track and choose an experiment

Linden Facilities needs 24 hourly predictions as one file within 60 seconds. Continue in the same repository and `work/`; the saved experiment plan and selected run carry into later sprints. This is an optional exploration notebook. [Campus lessons](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/README.md) also provide terminal/API routes. Keep one set of work. There is nothing to submit.

Follow [local setup](https://github.com/richhiey/ml-llm-ops/blob/main/docs/setup.md), select the project's Python 3.12 kernel, and run setup. Restarting a kernel loses variables, not saved run IDs. Do not reset the project between sections. Independent cells are for your attempt; guided outputs alone do not answer them.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Use Python 3.12. Locally run uv sync --locked --python 3.12 and select .venv/bin/python.')
# Hosted exploration is optional. Local setup uses the already installed editable project.
try:
    import linden_mlops
except ImportError:
    if 'google.colab' not in sys.modules:
        raise RuntimeError('Follow docs/setup.md from the repository root, then select its .venv kernel.')
    hosted_root = Path('/content/ml-llm-ops')
    if not hosted_root.exists():
        subprocess.run(['git', 'clone', 'https://github.com/richhiey/ml-llm-ops.git', str(hosted_root)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(hosted_root), 'pytest==8.4.2'], check=True)
    sys.path.insert(0, str(hosted_root / 'src'))
    import linden_mlops
from linden_mlops import (Config, Workspace, ingest, preprocess, train, evaluate, run_pipeline,
                          export_runs, verify_summary, register, approve, activate, batch_predict, reproduce)
from linden_mlops.lifecycle import ROOT, MODEL_NAME, code_revision
from linden_mlops.project import Project
from linden_mlops.dev import course_cli, project_tests, project_python, prepare_case
import pandas as pd
import numpy as np
progress = Project()
project = PACKAGE = ROOT
state = progress.work
ws = Workspace(progress.tracking)
print('Working source:', ROOT)
print('Retained work:', state)
print('Code:', code_revision())
print('Saved decisions:', sorted(progress.data))

## 1 · Why a list of scores stops working

Linden's planner sees “run 4: good”. Which source, data and configuration produced it? A run ID names one execution; parameters describe what ran, metrics describe measured behavior, and artifacts retain outputs. Logging them together makes comparison inspectable. A filename or rounded score cannot replace that link.

Use the Campus case to identify what is missing from an ad hoc note. No extra execution is needed.

## 2 · Decide what can be compared

The decision metric is validation `rmse_kwh` (lower is better), tied to historical error in the planning brief's units. MAE helps diagnose error, and autologged training scores describe fit on different rows. They do not replace held-out RMSE. Keep data bytes, split, feature order, training-only scaler fit, solver, guard and source fixed; vary `alpha` only. The reserved final rows remain closed until selection is fixed.

| Comparison | Interpretation |
|---|---|
| Same controls, different alpha | Controlled regularisation comparison |
| Different partition or source | Different question; exclude from this ranking |
| Same model, logging on/off | Logging inspection; not another alpha trial |

Predict why a lower training error does not by itself justify promotion.

## 3 · Declare five runs before seeing scores

Edit `configs/experiment.json` in this repository. Choose five distinct positive alpha values spanning weaker and stronger regularisation, including `1` as the comparison anchor. The supplied values are a starting range, not a required winning answer. In one short note state spacing rationale, fixed controls, why validation RMSE is primary, and an excluded example using different evaluation conditions. Do not execute that example or open the final holdout.

The next cell records your saved config. A completed matrix cannot silently become a different plan; use a deliberate new work directory only when you intend a new experiment.

In [ ]:
plan = json.loads((ROOT / 'configs/experiment.json').read_text())
assert (plan['primary_metric'], plan['direction'], plan['partition']) == ('rmse_kwh', 'minimize', 'validation')
print('Declared alphas:', progress.plan(plan['alphas']))

## 4 · Run the declared matrix and inspect tracking

`Workspace.run` opens a tracked execution, calls the pipeline, explicitly logs held-out metrics, and stores the fitted model and recovery recipe. Autologging adds estimator metadata during fitting. Each ID below must have one guarded validation configuration. The cell reopens a completed matrix rather than training it again.

In [ ]:
if not progress.data.get('matrix_run_ids'):
    records = [ws.run(Config(alpha=a, range_guard=True, autolog=True), name=f'alpha-{a}')
               for a in progress.require('planned_alphas')]
    progress.record_matrix(ws, records)
rows = progress.matrix(ws)
rows.to_csv(state / 'runs.csv', index=False)
display(rows.sort_values('alpha'))
print('Saved IDs:', progress.require('matrix_run_ids'))

Inspect one actual record next. A parameter answers what ran; `rmse_kwh` answers held-out prediction error; `reproduction/manifest.json` connects recipe and environment. The model URI identifies the fitted artifact. Look inside `src/linden_mlops/lifecycle.py` to find the calls that logged these fields.

In [ ]:
sample_id = progress.require('matrix_run_ids')[0]
raw = ws.client.get_run(sample_id)
print('Run:', sample_id)
print('Parameters:', raw.data.params)
print('Metrics:', raw.data.metrics)
print('Artifacts:', [a.path for a in ws.client.list_artifacts(sample_id)])
print('Recovery files:', [a.path for a in ws.client.list_artifacts(sample_id, 'reproduction')])
print('Local UI command: uv run linden-mlops ui --state', ws.path)

## 5 · Compare the five runs and save your choice

Open the printed command in a separate local terminal, then `http://127.0.0.1:5050`. Find `linden-demand`, select the five saved IDs, open their comparison and display `alpha` and `rmse_kwh`. Compare controls in the raw rows and identify the minimum. A hosted `localhost` does not expose a laptop UI; use local setup for this task.

Choose the actual ID yourself. In the cell below call `progress.select(ws, 'your actual ID')` after checking the UI and raw table. It validates the declared numeric rule and saves the full configuration for registry/recovery. It does not prove deployment cost or future accuracy.

In [ ]:
# Your choice: progress.select(ws, 'ACTUAL_RUN_ID')
# Explain the comparable controls and why this metric supports your choice.

In [ ]:
selection = progress.selected()
print('Retained selection:', selection)
(state / 'selected-config.json').write_text(json.dumps(selection['config'], indent=2) + '\n')

## 6 · Inspect what autologging adds

Keep the existing autologged alpha `1` run. Create one otherwise identical manual run only. The wrapper still logs validation RMSE, lineage, model and recovery files in both cases. Compare the key sets below: expect an estimator field such as `ridge__alpha` only with autologging. Verify validation RMSE agrees within `1e-6` kWh. Inspect why `log_models=False` avoids logging a second model automatically.

Explain convenience versus control using actual keys, not just the Boolean setting. Reopening the cell reuses both IDs.

In [ ]:
if not progress.data.get('logging_pair'):
    auto_id = rows.loc[rows.alpha.astype(float).eq(1), 'run_id'].iloc[0]
    manual = ws.run(Config(alpha=1, range_guard=True, autolog=False), name='manual-anchor')
    progress.save(logging_pair={'autolog': auto_id, 'manual': manual['run_id']})
pair = {mode: ws.client.get_run(rid) for mode, rid in progress.require('logging_pair').items()}
for mode, run in pair.items():
    print(mode, run.info.run_id, sorted(run.data.params), sorted(run.data.metrics))
assert abs(pair['autolog'].data.metrics['rmse_kwh'] - pair['manual'].data.metrics['rmse_kwh']) <= 1e-6
print('Added estimator keys:', sorted(set(pair['autolog'].data.params) - set(pair['manual'].data.params)))

## 7 · Check an actual assistant recommendation

Give your institution-approved assistant the five comparable raw rows and this request: “Recommend a run using lower validation rmse_kwh. Name the run ID, explain comparable controls, and separate measurements from claims needing other evidence.” Keep its actual response in your working notes. No API key or integration is required by this project.

Use `verify_summary(rows, proposed_id)` for the numeric recommendation. Break its prose into claims and mark each supported, inaccurate or unsupported with the relevant raw field. A correct ranking cannot establish cost, future generalisation or production readiness. If the response is accurate, record that; do not invent an error.

First observe a deliberately wrong numeric recommendation below. Expect `False`. The function also rejects incompatible evidence, which is different from a poor ranking.

In [ ]:
worst_id = rows.sort_values('rmse_kwh').iloc[-1].run_id
wrong = verify_summary(rows, worst_id)
print('Deliberately wrong advice:', wrong)
assert not wrong['recommendation_matches_metric']

In [ ]:
# Add the actual response's proposed ID and run verify_summary(rows, proposed_id).
# Keep one claim -> raw field -> supported/inaccurate/unsupported table.
# Qualify only actual issues; the response may be fully accurate.

## 8 · Harbor asks whether the recommendation is defensible

Use the existing experiment, logging pair, UI comparison and actual assistant review. Harbor asks: “Does your best historical validation score also establish the cheapest serving option?” Explain what the table proves and what cost evidence is absent. Confirm all five planned configurations appear once in the comparable set, your selected ID matches the primary metric, and explicit validation logging remains visible with autologging.

Reuse your existing claim table and selection. Do not rerun the matrix or create a separate report. The saved selection is the input to Sprint 3.